# AI Quant-Mate Session 7 & 8 — AI 스트레스 테스터 (MDD · VaR · 시나리오)

주제: **"내 포트폴리오는 얼마나 깨질까?"** — 평균 뒤에 숨은 꼬리를 수치화하고, 시나리오 충격을 AI에게 자연어로 의뢰한다.

## 흐름 (총 22단계)

**Phase E — 리스크 측정 기초**
1. DB 연결 + 시세 로딩 (3주차 `quant.db` 재사용)
2. NAV 시계열 만들기
3. MDD — `cummax` 한 함수
4. MDD 시각화 (NAV + drawdown)
5. 일일 수익률 분포 → Historical VaR
6. CVaR (Expected Shortfall)
7. Parametric VaR (정규분포 가정)
8. Monte Carlo VaR

**Phase F — 포트폴리오 위험**

9. 포트폴리오 비중 + 일일 수익률
10. 포트 VaR vs 종목별 VaR 합 (분산효과)
11. 상관관계 행렬
12. 포트 분포 시각화 + VaR 수직선

**Phase G — 시나리오 스트레스**

13. 베타 추정 (회귀)
14. 1요인 스트레스 (코스피 -15%)
15. 다요인 민감도 행렬

**Phase H — AI 연결**

16. Anthropic 클라이언트 준비
17. Function Calling 도구 정의 (`run_stress_test`)
18. 1차 호출 → tool_use 추출
19. 2차 호출 → AI 자연어 해설

**Phase I — 검증과 영속화**

20. VaR 백테스트 (Kupiec — 위반 카운트)
21. `stress_runs` 테이블 + INSERT
22. 백업 & 다음 주(5주차) 예고

> 이 노트북은 3주차에서 만든 `quant.db`를 입력으로 받는다. 같은 폴더에 `quant.db`가 없으면 Step 1에서 `pykrx`로 다시 적재한다.

## Step 0 — 라이브러리 install & import

이번 주차 신규: `numpy`(시뮬레이션·percentile), `scipy`(정규분포·회귀), `anthropic`(Function Calling). 1·2·3주차 친구들(`pandas`, `matplotlib`, `sqlite3`, `pykrx`)은 그대로 쓴다.

In [ ]:
!pip install pandas matplotlib numpy scipy anthropic pykrx -q

In [ ]:
import sqlite3
import shutil
import json
import os
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import norm
from pykrx import stock

---
## Step 1 — DB 연결 + 시세 로딩

3주차의 `quant.db`를 그대로 사용한다. `prices` 테이블에 5종목 × 약 23거래일이 들어 있어야 한다.

만약 비어 있으면 Step 1-b 셀이 자동으로 `pykrx`로 다시 채운다.

In [ ]:
conn = sqlite3.connect("quant.db")
conn.execute("PRAGMA foreign_keys = ON")
cur = conn.cursor()

tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table'", conn
)["name"].tolist()
print("📦 발견된 테이블:", tables)

if "prices" in tables:
    n = pd.read_sql_query("SELECT COUNT(*) AS n FROM prices", conn).iloc[0, 0]
    print(f"✅ prices 행 수: {n}")
else:
    print("⚠️ prices 테이블이 없다 — 다음 셀에서 새로 만든다")

**Step 1-b — prices가 비어 있으면 다시 적재**

3주차 노트북을 안 돌렸거나 DB가 없는 환경에서도 끝까지 동작하도록 fallback을 둔다.

In [ ]:
tickers_seed = {
    "005930": ("삼성전자",   "반도체",     "KOSPI"),
    "000660": ("SK하이닉스", "반도체",     "KOSPI"),
    "042700": ("한미반도체", "반도체장비", "KOSPI"),
    "035420": ("NAVER",      "인터넷",     "KOSPI"),
    "051910": ("LG화학",     "화학",       "KOSPI"),
}

# tickers / prices 보장
cur.execute("""
    CREATE TABLE IF NOT EXISTS tickers (
        ticker TEXT PRIMARY KEY, name TEXT NOT NULL, sector TEXT, market TEXT
    )
""")
cur.execute("""
    CREATE TABLE IF NOT EXISTS prices (
        ticker TEXT NOT NULL, date TEXT NOT NULL,
        open REAL, high REAL, low REAL, close REAL, volume INTEGER,
        PRIMARY KEY (ticker, date),
        FOREIGN KEY (ticker) REFERENCES tickers(ticker)
    )
""")
conn.commit()

with conn:
    conn.executemany(
        "INSERT OR REPLACE INTO tickers VALUES (?, ?, ?, ?)",
        [(t, n, s, m) for t, (n, s, m) in tickers_seed.items()],
    )

n_rows = pd.read_sql_query("SELECT COUNT(*) AS n FROM prices", conn).iloc[0, 0]

if n_rows < 50:
    # 충분한 시계열 길이를 위해 캘린더 1년
    end = datetime.today().date()
    start = end - timedelta(days=365)
    start_s, end_s = start.strftime("%Y%m%d"), end.strftime("%Y%m%d")

    rows = []
    for t in tickers_seed:
        ohlcv = stock.get_market_ohlcv_by_date(start_s, end_s, t)
        for d, r in ohlcv.iterrows():
            rows.append((
                t, d.strftime("%Y-%m-%d"),
                float(r["시가"]), float(r["고가"]), float(r["저가"]),
                float(r["종가"]), int(r["거래량"]),
            ))
    with conn:
        conn.executemany(
            "INSERT OR REPLACE INTO prices (ticker, date, open, high, low, close, volume) VALUES (?, ?, ?, ?, ?, ?, ?)",
            rows,
        )
    print(f"✅ pykrx로 {len(rows)}행 적재")
else:
    print(f"ℹ️ prices에 이미 {n_rows}행 존재 — 그대로 사용")

# 종가 시계열 피벗 (행=날짜, 열=종목)
prices = pd.read_sql_query(
    "SELECT date, ticker, close FROM prices ORDER BY date ASC", conn,
).pivot(index="date", columns="ticker", values="close")
prices.index = pd.to_datetime(prices.index)
print("📊 prices shape:", prices.shape)
prices.tail()

---
## Step 2 — NAV 시계열 만들기

5종목을 동일가중(20%씩)으로 보유했다고 가정하고, 첫날을 1.0으로 정규화한 NAV(누적자산가치) 곡선을 만든다.

- 종목별 가격 → 첫날 100% 정규화
- 동일가중 평균 → 포트폴리오 NAV
- 이후 모든 위험지표(MDD·VaR·시나리오)가 이 NAV에서 출발한다.

In [ ]:
tickers_list = list(prices.columns)
weights = pd.Series([0.20] * len(tickers_list), index=tickers_list)
print("⚖️ 비중:", weights.to_dict())

# 종목별 첫날 100% 정규화
norm_prices = prices / prices.iloc[0]
nav = (norm_prices * weights).sum(axis=1)   # 동일가중 합 = 포트 NAV
nav.name = "NAV"

print(f"기간: {nav.index[0].date()} ~ {nav.index[-1].date()} ({len(nav)} 거래일)")
print(f"최종 NAV: {nav.iloc[-1]:.4f}  (총 수익률 {nav.iloc[-1]-1:+.2%})")
nav.tail()

---
## Step 3 — MDD: `cummax` 한 함수가 핵심

\\[ \text{MDD} = \min_t \dfrac{V_t - \max_{s \le t} V_s}{\max_{s \le t} V_s} \\]

코드는 한 줄짜리 트릭으로 끝난다.

- `nav.cummax()` — 지금까지의 최고점(봉우리들의 누적)
- `nav / peak - 1` — 현재 낙폭(음수). 이게 곧 drawdown 시계열
- `.min()` — 가장 깊은 골 = MDD

In [ ]:
peak     = nav.cummax()
drawdown = nav / peak - 1

mdd       = drawdown.min()
mdd_date  = drawdown.idxmin()
peak_date = nav.loc[:mdd_date].idxmax()

print(f"📉 MDD       = {mdd:.2%}")
print(f"   peak 일자  = {peak_date.date()}  (NAV {nav.loc[peak_date]:.4f})")
print(f"   trough 일자 = {mdd_date.date()}  (NAV {nav.loc[mdd_date]:.4f})")

---
## Step 4 — MDD 시각화

NAV(파랑)와 drawdown(빨강 음수)을 같은 그림에 그린다. 봉우리 뒤마다 빨간 골짜기가 보인다.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(11, 5.5), sharex=True)

axes[0].plot(nav.index, nav.values, color="#00f2ff", linewidth=1.6, label="NAV")
axes[0].plot(nav.index, peak.values, color="#888", linewidth=0.8, linestyle="--", label="Running peak")
axes[0].axvline(mdd_date, color="#ff4757", linewidth=0.8, alpha=0.6)
axes[0].set_title("포트폴리오 NAV (동일가중)")
axes[0].grid(alpha=0.3); axes[0].legend()

axes[1].fill_between(drawdown.index, drawdown.values, 0, color="#ff4757", alpha=0.4)
axes[1].axhline(mdd, color="#ff4757", linestyle="--", linewidth=0.8, label=f"MDD {mdd:.2%}")
axes[1].set_title("Drawdown")
axes[1].grid(alpha=0.3); axes[1].legend()

plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

---
## Step 5 — 일일 수익률 분포 → Historical VaR

> "95% 신뢰수준 1일 VaR이 -2.4%다" = "정상적인 100거래일 중 95일은 -2.4%보다 덜 잃는다."
> 나머지 5일은 -2.4%보다 **더** 잃을 수 있다. VaR은 경계선이지 한계가 아니다.

Historical VaR은 가장 정직한 방법 — 분포 가정 없이 과거 데이터의 5% 분위수를 그대로 사용한다.

In [ ]:
port_ret = nav.pct_change().dropna()

var_95_hist  = np.percentile(port_ret, 5)
var_99_hist  = np.percentile(port_ret, 1)

print(f"📉 Historical VaR 95% (1d) = {var_95_hist:.2%}")
print(f"📉 Historical VaR 99% (1d) = {var_99_hist:.2%}")
print(f"   (총 {len(port_ret)}거래일 기준)")

---
## Step 6 — CVaR (Expected Shortfall)

VaR이 **경계선**이라면 CVaR은 **그 너머의 평균**. 꼬리의 두께를 본다.

같은 VaR을 가진 두 자산도 CVaR은 다를 수 있다. 진짜 위험은 CVaR에서 드러난다.

In [ ]:
cvar_95 = port_ret[port_ret <= var_95_hist].mean()
cvar_99 = port_ret[port_ret <= var_99_hist].mean()

print(f"   Historical CVaR 95% = {cvar_95:.2%}")
print(f"   Historical CVaR 99% = {cvar_99:.2%}")
print(f"   → VaR 너머 5% 영역의 평균 손실이다")

# 비교 표
pd.DataFrame({
    "VaR":  [var_95_hist, var_99_hist],
    "CVaR": [cvar_95, cvar_99],
}, index=["95%", "99%"]).style.format("{:.2%}")

---
## Step 7 — Parametric VaR (정규분포 가정)

\\[ \text{VaR}_{95\%} = \mu - 1.645 \sigma \\]

`norm.ppf(0.05) ≈ -1.645`. 한 줄짜리 닫힌 공식이다.

함정: 실제 주식 수익률은 정규분포보다 **꼬리가 두껍다(fat tail)**. 보통 Parametric이 Historical보다 작게(덜 보수적으로) 나온다.

In [ ]:
mu    = port_ret.mean()
sigma = port_ret.std()

var_95_param = mu + sigma * norm.ppf(0.05)
var_99_param = mu + sigma * norm.ppf(0.01)

print(f"   μ = {mu:.4%},  σ = {sigma:.4%}")
print(f"   Parametric VaR 95% = {var_95_param:.2%}  (Historical: {var_95_hist:.2%})")
print(f"   Parametric VaR 99% = {var_99_param:.2%}  (Historical: {var_99_hist:.2%})")
print()
print("⚠️ 두 값의 차이 = 분포가 정규에서 얼마나 벗어났는가의 신호")

---
## Step 8 — Monte Carlo VaR

"가능한 미래 N개"를 시뮬레이션해서 분포로 본다. 다자산·비선형 상품에서 위력.

여기선 가장 단순한 형태: 정규분포에서 1만 번 추출.

In [ ]:
np.random.seed(42)   # 재현성

N = 10_000
sims = np.random.normal(mu, sigma, N)

var_95_mc  = np.percentile(sims, 5)
cvar_95_mc = sims[sims <= var_95_mc].mean()

print(f"   MC VaR  95% = {var_95_mc:.2%}")
print(f"   MC CVaR 95% = {cvar_95_mc:.2%}")
print()

# 3종 비교
comparison = pd.DataFrame({
    "VaR 95%":  [var_95_hist, var_95_param, var_95_mc],
    "VaR 99%":  [var_99_hist, var_99_param, np.percentile(sims, 1)],
}, index=["Historical", "Parametric", "Monte Carlo"])
print("📊 3종 VaR 비교:")
comparison.style.format("{:.2%}")

---
## Step 9 — 포트폴리오 비중 + 일일 수익률 행렬

지금까지는 동일가중 포트의 NAV에서 바로 수익률을 뽑았다. 이제 명시적으로 **종목별 수익률 행렬 × 비중 벡터**의 형태로 다시 짠다 — 다음 단계에서 분산효과를 보기 위해.

In [ ]:
ret_mat = prices.pct_change().dropna()    # 행=날짜, 열=종목

# numpy 배열로
W   = weights.values
R   = ret_mat.values
port_ret_vec = R @ W                       # 행렬곱: shape = (T,)

# 검증: NAV에서 바로 뽑은 수익률과 거의 같아야 한다
diff = pd.Series(port_ret_vec, index=ret_mat.index) - port_ret
print(f"   max |차이|: {diff.abs().max():.2e}  (≈ 0 이어야 정상)")
print(f"   포트 일일 수익률 평균 = {port_ret_vec.mean():.4%}")
print(f"   포트 일일 수익률 표준편차 = {port_ret_vec.std():.4%}")

---
## Step 10 — 분산효과: 포트 VaR vs 종목별 VaR 합

개별 종목 VaR을 그냥 더한 값보다 **포트 VaR이 작다**. 그 차이가 분산효과(diversification benefit).

이유: 종목들이 100% 같이 움직이지 않기 때문(상관계수 < 1).

In [ ]:
# 종목별 VaR 95%
indiv_var = ret_mat.apply(lambda x: np.percentile(x, 5))

# "비중을 곱해 더한" 단순 합산 (분산효과 없을 때의 가상 VaR)
naive_sum = (indiv_var * weights).sum()

# 실제 포트 VaR
port_var_95 = np.percentile(port_ret_vec, 5)

print("📊 종목별 VaR 95%:")
for t, v in indiv_var.items():
    print(f"   {t}: {v:.2%}")
print()
print(f"   ① 비중가중 단순 합:  {naive_sum:.2%}")
print(f"   ② 실제 포트 VaR:     {port_var_95:.2%}")
print(f"   분산효과(② − ①):    {port_var_95 - naive_sum:+.2%}  ← 양수일수록 덜 위험")

---
## Step 11 — 상관관계 행렬

**분산효과는 ρ가 만든다.** 그런데 위기에는 ρ → 1로 수렴한다 — 분산은 가장 필요할 때 사라진다.

평시 ρ ≈ 0.3, 위기 ρ → 0.9. 이게 오늘의 가장 중요한 경고.

In [ ]:
corr = ret_mat.corr()
print("📊 상관계수 행렬:")
print(corr.round(2))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(corr.values, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr))); ax.set_yticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)
for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.values[i, j]) > 0.5 else "black", fontsize=10)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title("상관계수 행렬 (일일 수익률)")
plt.tight_layout(); plt.show()

---
## Step 12 — 포트 분포 시각화 + VaR 수직선

숫자만 던지면 머리에 안 박힌다. 분포에 빨간 수직선과 음영을 같이 그려야 "꼬리"라는 단어가 살아난다.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.6))

ax.hist(port_ret_vec, bins=60, color="#00f2ff", alpha=0.6, edgecolor="#0a0e14")
ax.axvline(port_var_95, color="#ff4757", linewidth=2, label=f"VaR 95% = {port_var_95:.2%}")
ax.axvspan(port_ret_vec.min(), port_var_95, color="#ff4757", alpha=0.15, label="CVaR 영역 (좌측 5%)")
ax.axvline(port_ret_vec.mean(), color="#adff2f", linestyle="--", linewidth=1, label=f"평균 {port_ret_vec.mean():+.2%}")

# 정규분포 곡선 겹쳐 그리기
x = np.linspace(port_ret_vec.min(), port_ret_vec.max(), 200)
y = norm.pdf(x, mu, sigma)
ax2 = ax.twinx()
ax2.plot(x, y, color="#ffa502", linewidth=1.5, alpha=0.8)
ax2.set_yticks([])

ax.set_title("포트 일일 수익률 분포 + VaR 95%")
ax.set_xlabel("Daily return"); ax.set_ylabel("Frequency")
ax.legend(loc="upper left")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

---
## Step 13 — 베타 추정 (회귀)

**베타 = 시장 수익률 1% 변할 때 종목 수익률은 몇 % 변하는가.** CAPM에서 온 개념.

여기선 KOSPI 지수를 시장 프록시로 사용. `pykrx`의 `get_index_ohlcv_by_date`로 받는다.

In [ ]:
# KOSPI 지수를 시장 프록시로 사용 — pykrx 응답이 막히면 합성 시장으로 fallback.
end = ret_mat.index.max().date()
start = ret_mat.index.min().date()

kospi_ret = None
try:
    kospi = stock.get_index_ohlcv_by_date(
        start.strftime("%Y%m%d"), end.strftime("%Y%m%d"), "1001"
    )
    if kospi is not None and not kospi.empty and "종가" in kospi.columns:
        kospi.index = pd.to_datetime(kospi.index)
        kospi_ret = kospi["종가"].pct_change().dropna()
        kospi_ret.name = "KOSPI"
        market_label = "KOSPI"
    else:
        raise ValueError("pykrx returned empty or malformed KOSPI frame")
except Exception as e:
    print(f"⚠️ KOSPI 지수 응답 실패 ({e}) — 합성 시장(5종목 동일가중 평균)으로 fallback")
    kospi_ret = ret_mat.mean(axis=1)
    kospi_ret.name = "MARKET"
    market_label = "MARKET (synthetic)"

# 종목별 수익률과 시장 수익률을 같은 인덱스로 정렬
joined = ret_mat.join(kospi_ret, how="inner")
mkt_col = kospi_ret.name

betas = {}
for t in tickers_list:
    slope, intercept, r, p, se = stats.linregress(joined[mkt_col], joined[t])
    betas[t] = slope
betas = pd.Series(betas, name="beta")

print(f"📊 종목별 시장 베타 (시장={market_label}):")
print(betas.round(3))

---
## Step 14 — 1요인 스트레스 (코스피 -15%)

가장 단순한 시나리오: **베타 × 시장충격 × 비중**.

이 한 줄이 모든 1요인 스트레스 모델의 뼈대.

In [ ]:
def one_factor_stress(market_shock: float) -> dict:
    """코스피 시장충격을 베타로 종목별 손익으로 환산하고 포트 손익을 합산."""
    stock_pnl = betas * market_shock          # Series
    port_pnl  = (stock_pnl * weights).sum()
    return {
        "market_shock": market_shock,
        "stock_pnl":    stock_pnl.round(4).to_dict(),
        "port_pnl":     float(round(port_pnl, 4)),
    }

result_15 = one_factor_stress(-0.15)
print(f"📉 코스피 -15% 시 포트 손익: {result_15['port_pnl']:+.2%}")
print("   종목별 손익:")
for t, v in result_15["stock_pnl"].items():
    print(f"     {t}: {v:+.2%}")

---
## Step 15 — 다요인 민감도 행렬

**민감도 행렬 × 충격 벡터 = 종목별 손익.** 모든 위험관리 시스템의 뼈대.

행 = 종목, 열 = 위험요인. 셀 = "그 요인 +1%당 종목 수익률 변화."

여기선 시장 베타(Step 13)는 회귀로 추정하고, 금리·환율 민감도는 **시연을 위한 가정값**으로 둔다. 실전에선 다중회귀로 같이 추정한다.

In [ ]:
# 시연용 가정 — 실전은 다중회귀로 추정한다
sensitivities = pd.DataFrame({
    "market": betas.values,
    "rate":   [-0.20, -0.30, 0.05, -0.10, -0.15],   # 금리 +1%당 수익률 변화
    "fx_usd": [ 0.40,  0.10, 0.50, -0.20,  0.05],   # USD/KRW +1%당
}, index=tickers_list)

print("📊 민감도 행렬:")
print(sensitivities.round(3))

def multi_factor_stress(market_shock=0.0, rate_shock=0.0, fx_shock=0.0) -> dict:
    shocks = pd.Series({"market": market_shock, "rate": rate_shock, "fx_usd": fx_shock})
    stock_pnl = sensitivities @ shocks                 # Series
    port_pnl  = float((stock_pnl * weights).sum())
    worst_idx = stock_pnl.idxmin()
    return {
        "shocks":         shocks.round(4).to_dict(),
        "stock_pnl":      stock_pnl.round(4).to_dict(),
        "port_pnl":       round(port_pnl, 4),
        "worst_ticker":   worst_idx,
        "worst_pnl":      float(round(stock_pnl.min(), 4)),
    }

# 시연: 금리 +200bp, 코스피 -10%, USD +5%
demo = multi_factor_stress(market_shock=-0.10, rate_shock=0.02, fx_shock=0.05)
print()
print(f"📉 시나리오 (시장 -10%, 금리 +200bp, USD +5%):")
print(f"   포트 손익      = {demo['port_pnl']:+.2%}")
print(f"   가장 취약 종목 = {demo['worst_ticker']} ({demo['worst_pnl']:+.2%})")

---
## Step 16 — Anthropic 클라이언트 준비

자연어 시나리오 의뢰를 받기 위해 Claude(Anthropic) SDK를 쓴다. API 키가 없으면 다음 셀들에서 더미 응답으로 대체된다.

In [ ]:
# --- Colab ---
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

try:
    import anthropic
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY) if ANTHROPIC_API_KEY else None
except ImportError:
    client = None

print("✅ Anthropic client 준비" if client else "⚠️ API 키 없음 — 더미 응답으로 진행")

---
## Step 17 — Function Calling 도구 정의

AI에게 호출 가능한 함수 명세(`input_schema`)를 미리 알려 준다. JSON Schema 형식.

**description이 핵심.** 모호하면 AI가 도구를 안 부르거나 엉뚱한 값을 넣는다.

In [ ]:
TOOLS = [
    {
        "name": "run_stress_test",
        "description": (
            "포트폴리오에 거시 시나리오 충격을 적용한 손익을 계산한다. "
            "시장(코스피), 금리, USD/KRW 환율의 충격을 받아 종목별·포트 단위 손익을 반환한다. "
            "각 충격은 비율 단위 — 예: market_shock=-0.10 은 코스피 -10%."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "market_shock": {"type": "number", "description": "코스피 충격 (예: -0.15 = -15%)"},
                "rate_shock":   {"type": "number", "description": "금리 변화 (예: 0.02 = +200bp)"},
                "fx_shock":     {"type": "number", "description": "USD/KRW 변화율 (예: 0.05 = +5%)"},
            },
            "required": ["market_shock"],
        },
    }
]

print("✅ tools 정의 완료 — 1개 도구")
print(json.dumps(TOOLS, indent=2, ensure_ascii=False)[:500] + " ...")

---
## Step 18 — 1차 호출: tool_use 추출

자연어 질문을 받아 AI가 어떤 인자로 도구를 부를지 결정하게 한다.

`stop_reason == "tool_use"` 이면 마지막 content가 ToolUseBlock — 그 `.input`이 우리에게 필요한 dict.

In [ ]:
user_question = "금리 +200bp, 코스피 -10%, USD/KRW +5% 시나리오로 내 포트 영향을 분석해줘"

if client is None:
    # 더미: AI가 인자를 추출했다고 가정
    tool_input = {"market_shock": -0.10, "rate_shock": 0.02, "fx_shock": 0.05}
    tool_use_id = "dummy_tool_use_id"
    first_resp_content = None
    print("⚠️ 더미 모드 — 미리 정의된 인자로 진행")
else:
    resp = client.messages.create(
        model="claude-sonnet-4-6",
        max_tokens=1024,
        tools=TOOLS,
        messages=[{"role": "user", "content": user_question}],
    )
    print("stop_reason:", resp.stop_reason)
    if resp.stop_reason == "tool_use":
        tool_block = next(b for b in resp.content if b.type == "tool_use")
        tool_input  = tool_block.input
        tool_use_id = tool_block.id
        first_resp_content = resp.content
    else:
        # 도구를 안 부른 경우 — 텍스트만 출력
        tool_input  = None
        tool_use_id = None
        first_resp_content = resp.content
        print("ℹ️ AI가 도구를 호출하지 않았다 — 응답:")
        print(resp.content[0].text)

print("🛠️ AI가 추출한 인자:", tool_input)

---
## Step 19 — 우리 함수 실행 + 2차 호출 (자연어 해설)

AI가 제안한 인자로 우리 `multi_factor_stress`를 실행하고, 그 결과를 다시 AI에게 넘겨 자연어로 해설받는다.

핵심: **계산은 우리 코드가, 해석은 AI가.**

In [ ]:
SYSTEM_PROMPT = """너는 보수적인 리스크 매니저다. 다음 원칙을 지킨다:
- 숫자는 결과 dict의 값을 본문에 그대로 인용한다 (% 둘째 자리까지).
- "최악의 종목 → 이유 → 즉시 줄여야 할 비중" 순서로 답한다.
- 데이터에 없는 추측은 하지 않는다. 모르면 "데이터 부족"이라 답한다."""

if tool_input is not None:
    tool_result = multi_factor_stress(**tool_input)
    print("🧮 계산 결과:")
    print(json.dumps(tool_result, indent=2, ensure_ascii=False))

    if client is None:
        ai_summary = (
            "[더미 응답]\n"
            f"포트 손익: {tool_result['port_pnl']:+.2%}\n"
            f"가장 취약: {tool_result['worst_ticker']} ({tool_result['worst_pnl']:+.2%})\n"
            "헤지 아이디어: 가장 취약한 종목의 비중을 절반으로 축소하고, 현금 비중을 그만큼 늘린다."
        )
    else:
        followup = client.messages.create(
            model="claude-sonnet-4-6",
            max_tokens=1024,
            system=SYSTEM_PROMPT,
            tools=TOOLS,
            messages=[
                {"role": "user",      "content": user_question},
                {"role": "assistant", "content": first_resp_content},
                {"role": "user", "content": [{
                    "type": "tool_result",
                    "tool_use_id": tool_use_id,
                    "content": json.dumps(tool_result, ensure_ascii=False),
                }]},
            ],
        )
        ai_summary = followup.content[0].text

    print("\n🤖 AI 해설:\n")
    print(ai_summary)
else:
    ai_summary = None
    tool_result = None
    print("⚠️ tool_input이 없어 2차 호출을 건너뛴다")

---
## Step 20 — VaR 백테스트 (Kupiec — 위반 카운트)

"내 VaR 모델이 정말 95%인가?" 과거 데이터로 위반 횟수를 직접 센다.

- 위반 ≈ 5% → 모델 적정
- 위반 ≫ 5% → VaR 과소추정 (정규분포 가정 깨짐)
- 위반 ≪ 5% → 보수적이지만 자본 비효율

여기선 rolling window 60일로 매일 VaR을 다시 추정해 다음 날 손실과 비교한다.

In [ ]:
window = 60

rolling_var = port_ret.rolling(window).quantile(0.05).shift(1)   # 어제 추정한 VaR
backtest = pd.concat([port_ret.rename("real"), rolling_var.rename("var95")], axis=1).dropna()

backtest["violation"] = backtest["real"] < backtest["var95"]
n_total      = len(backtest)
n_violations = int(backtest["violation"].sum())
expected     = n_total * 0.05

print(f"📊 백테스트 (rolling {window}일 Historical VaR 95%):")
print(f"   거래일 수      = {n_total}")
print(f"   실제 위반      = {n_violations}회")
print(f"   기대 위반      = {expected:.1f}회 (5%)")
print(f"   위반률         = {n_violations / n_total:.2%}")

verdict = "적정" if abs(n_violations - expected) <= max(2, 0.4 * expected) else "재검토 필요"
print(f"   판정          = {verdict}")

---
## Step 21 — `stress_runs` 테이블 + INSERT

오늘 돌린 스트레스 결과를 DB에 누적해 둔다. 같은 시나리오를 매주 돌려 쌓으면 **취약성 시계열**이 만들어진다.

3주차 패턴 그대로: CREATE → INSERT → SELECT.

In [ ]:
cur.execute("""
    CREATE TABLE IF NOT EXISTS stress_runs (
        id              INTEGER PRIMARY KEY AUTOINCREMENT,
        run_date        TEXT    NOT NULL,
        scenario_name   TEXT    NOT NULL,
        params          TEXT    NOT NULL,
        port_pnl        REAL,
        worst_ticker    TEXT,
        worst_pnl       REAL,
        ai_summary      TEXT
    )
""")
conn.commit()

today_iso = datetime.today().strftime("%Y-%m-%d")

if tool_input is not None and tool_result is not None:
    with conn:
        conn.execute(
            """INSERT INTO stress_runs
               (run_date, scenario_name, params, port_pnl, worst_ticker, worst_pnl, ai_summary)
               VALUES (?, ?, ?, ?, ?, ?, ?)""",
            (
                today_iso,
                "rate_up_market_down",
                json.dumps(tool_input, ensure_ascii=False),
                tool_result["port_pnl"],
                tool_result["worst_ticker"],
                tool_result["worst_pnl"],
                ai_summary,
            ),
        )
    print(f"✅ stress_runs 적재 완료 ({today_iso})")

pd.read_sql_query(
    """SELECT id, run_date, scenario_name, port_pnl, worst_ticker, worst_pnl,
              substr(ai_summary, 1, 60) AS summary_preview
       FROM stress_runs ORDER BY id DESC LIMIT 5""",
    conn,
)

---
## Step 22 — 백업 & 다음 주 예고

**Week 5 (Session 9 & 10): "포트폴리오 옵티마이저"**
- 위험을 측정했으니 다음은 **최적화** — Markowitz 평균-분산, Sharpe, 블랙리터만.
- 4주차의 VaR·시나리오 결과를 **제약조건**으로 쓰는 위험 한도형 최적화.
- AI의 "직관"을 어떻게 끼워 넣을 것인가.

## 핵심 메시지
1. 리스크는 평균이 아니라 **꼬리**다. MDD·VaR·CVaR로 그 꼬리를 수치화한다.
2. 통계만으로는 부족하다. **시나리오 = 이야기**가 통계의 빈 칸을 메운다.
3. 분업의 원칙: **계산은 코드**, **해석은 AI**, **판단은 사람**.
4. 모든 모델은 가정 위에 선다. 가정이 깨지는 그날을 **가정 자체로** 검증한다.

In [ ]:
stamp = datetime.today().strftime("%Y%m%d")

conn.commit()
shutil.copyfile("quant.db", f"quant_backup_{stamp}.db")

# 오늘의 위험 요약 CSV
risk_summary = pd.DataFrame({
    "metric": ["MDD", "VaR 95% (Hist)", "VaR 95% (Param)", "VaR 95% (MC)",
               "CVaR 95% (Hist)", "Stress (시장-10%/금리+200bp/USD+5%)"],
    "value":  [mdd, var_95_hist, var_95_param, var_95_mc, cvar_95,
               tool_result["port_pnl"] if tool_result else None],
})
risk_summary.to_csv(f"risk_summary_{stamp}.csv", index=False, encoding="utf-8-sig")

print(f"✅ DB 백업: quant_backup_{stamp}.db")
print(f"✅ 위험 요약: risk_summary_{stamp}.csv")
print()
print(risk_summary.to_string(index=False))

conn.close()
print("\n✅ DB 연결 종료")